## 1. Setup

Install from the repository root with `python -m pip install -r requirement.txt`, then select that environment as the notebook kernel. Restart the kernel after changing CAAF. This notebook imports the current repository sources and performs no file exports.

Preprocessing always uses the full demonstration data. The Ranking section provides a separate commented smoke-test call; skip the full ranking cell and uncomment the smoke-test cell when you want a short verification run.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the repository independently of the current notebook directory.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "CAAF" / "__init__.py").is_file() and (p / "Demo").is_dir()), None)
if root is None:
    raise FileNotFoundError("Run this notebook from the CAAF repository or its Demo directory.")
sys.path.insert(0, str(root))
import CAAF

from Demo.utils.shm_data_processing import generate_shm_data

## 2. Preprocessing

Generate the first three cantilever bending modes at 30 candidate positions from `x/L = 1/30` to `1`. Normalize each mode by its Euclidean norm, then combine 20, 50, and 50 uniformly spaced modal coordinates in `[-1, 1]` to obtain 50,000 displacement samples. Divide all displacements by their global maximum absolute value. CAAF receives these scaled inputs with `normalization="none"`; targets are the three modal coordinates.

In [ ]:
# Generate all 50,000 cases without reducing preprocessing for smoke testing.
X, y, positions, mode_shapes = generate_shm_data()
assert X.shape == (50000, 30) and y.shape == (50000, 3)
assert np.isfinite(X).all() and np.isfinite(y).all()
print(f"Displacement shape: {X.shape}; modal-coordinate shape: {y.shape}")

## 3. Ranking

AP groups correlated displacement histories. The MLP predicts three modal coordinates, and mean absolute IG weights all three outputs equally. The zero baseline is the undeformed beam. Five runs use independent initializations; the final five positions are computed from their averaged attribution scores.


The next cell runs full ranking. For a smoke test, skip it and uncomment every line of the following cell, then continue to the results table and plot. Smoke testing preserves all preprocessed data and clustering; it reduces only training and attribution effort. `verbose=True` shows each processing stage; set it to `False` to silence CAAF progress.

Normalization and clustering use all supplied inputs. Validation loss selects the model checkpoint and is not an independent predictive test score. These demos use current CAAF models and splitting rules rather than reproducing historical numerical rankings exactly.

In [ ]:
# Rank five sensor positions by their attribution to all three modal coordinates.
sensor_indices, percentages = CAAF.rank_sensors(
    X, y, n_sensors=5, normalization="none",
    clustering={"name": "ap", "preference": 0.991, "damping": 0.5,
                "max_iter": 10000, "convergence_iter": 10},
    model={"name": "mlp", "hidden_sizes": (32, 32, 32),
           "activation": "relu", "batch_norm": True},
    training={"epochs": 50, "n_runs": 5, "batch_size": 32,
              "optimizer": "adam", "lr": 1e-5, "weight_decay": 1e-2,
              "dtype": "float64", "scheduler": {"patience": 10, "eps": 1e-7}},
    ig={"baseline": "zero", "n_steps": 50, "max_samples": 20000,
        "aggregation": "mean_abs"},
    random_state=147, device="auto", verbose=True, return_percentages=True,
)

assert len(sensor_indices) == 5 and len(np.unique(sensor_indices)) == 5
assert np.isfinite(percentages).all() and (percentages >= 0).all()

In [ ]:
# # Rank five sensor positions by their attribution to all three modal coordinates.
# sensor_indices, percentages = CAAF.rank_sensors(
#     X, y, n_sensors=5, normalization="none",
#     clustering={"name": "ap", "preference": 0.991, "damping": 0.5,
#                 "max_iter": 10000, "convergence_iter": 10},
#     model={"name": "mlp", "hidden_sizes": (32, 32, 32),
#            "activation": "relu", "batch_norm": True},
#     training={"epochs": 2, "n_runs": 1, "batch_size": 32,
#               "optimizer": "adam", "lr": 1e-5, "weight_decay": 1e-2,
#               "dtype": "float64", "scheduler": {"patience": 10, "eps": 1e-7}},
#     ig={"baseline": "zero", "n_steps": 3, "max_samples": 512,
#         "aggregation": "mean_abs"},
#     random_state=147, device="auto", verbose=True, return_percentages=True,
# )
# 
# assert len(sensor_indices) == 5 and len(np.unique(sensor_indices)) == 5
# assert np.isfinite(percentages).all() and (percentages >= 0).all()

## 4. Results table

Indices are zero-based original sensor columns. Percentages retain their share of attribution across all cluster representatives, so selected values need not sum to 100%.

In [ ]:
# Report original zero-based sensor indices and normalized beam positions.
results = pd.DataFrame({"Rank": np.arange(1, len(sensor_indices) + 1),
                        "Sensor index": sensor_indices, "x/L": positions[sensor_indices],
                        "Attribution (%)": percentages})
results.style.hide(axis="index").format({"x/L": "{:.4f}", "Attribution (%)": "{:.3f}"})

## 5. Final sensor configuration

Sensor locations and rank labels below are computed from the selected ranking.

In [ ]:
# Draw the clamped beam, candidate positions, and computed sensor ranks.
fig, ax = plt.subplots(figsize=(11, 3), layout="constrained")
ax.plot([0, 1], [0, 0], color="0.25", linewidth=3)
ax.axvspan(-0.035, 0, color="0.8", hatch="///")
ax.scatter(positions, np.zeros_like(positions), color="0.65", s=18, label="Candidates", zorder=3)
selected = positions[sensor_indices]
ax.scatter(selected, np.zeros_like(selected), color="tab:red", marker="x", s=85, label="Selected sensors", zorder=4)
for rank, position in enumerate(selected, start=1):
    ax.annotate(str(rank), (position, 0), xytext=(0, 24 + 16 * ((rank - 1) % 2)),
                textcoords="offset points", ha="center", arrowprops={"arrowstyle": "-", "color": "0.5"})
ax.set(xlabel="x/L", xlim=(-0.04, 1.04), ylim=(-0.15, 0.3), yticks=[])
ax.legend(loc="lower right")
plt.show()